In [1]:
# Regrid population data to match 0.1x0.1 degrees

In [2]:
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

In [ ]:
def expand_grid(pop_orig):
    # === Define resolution and bounds ===
    lat_res = 1 / 120  # 0.008333... degrees
    lon_res = 1 / 120

    # Generate full latitude and longitude ranges
    lat_full = np.arange(-90 + lat_res / 2, 90, lat_res)
    lon_full = np.arange(-180 + lon_res / 2, 180, lon_res)

    # Confirm lengths match your original grid
    print(f"Longitude points for pop_orig: {len(pop_orig.lon)}, Longitude points: {len(lon_full)}")  # Should match lon: 43200

    # === Create new empty DataArray with NaNs ===
    pop_full = xr.DataArray(
        data=np.full((len(lat_full), len(lon_full)), np.nan),
        coords={"lat": lat_full, "lon": lon_full},
        dims=["lat", "lon"],
        name="population"
    )

    # === Add population data to empty DataArray ===
    # Find indices where original lat/lon match the full grid
    lat_idx = np.searchsorted(lat_full, pop_orig.lat.values)
    lon_idx = np.searchsorted(lon_full, pop_orig.lon.values)

    # Use advanced indexing to insert your data
    pop_full.values[np.ix_(lat_idx, lon_idx)] = pop_orig.values

    assert np.allclose(
        pop_full.sel(
            lat=pop_orig.lat, lon=pop_orig.lon, method="nearest", tolerance=1e-5
        ),
        pop_orig,
        equal_nan=True
    )

    return pop_full

In [7]:
# Regrid each population year [2000, 2010, 2020, ..., 2100]
for year in range(2000, 2101, 10):
    print(f"Processing {year}")
    if year == 2000:
        pop = xr.open_dataset(f"{POP_DIR}baseYr_total_{year}.nc4")["Band1"]
    else:
        pop = xr.open_dataset(f"{POP_DIR}ssp2_total_{year}.nc4")["Band1"]

    pop_full = expand_grid(pop)

    # From GBD “Aggregation to each 0.1 × 0.1 grid cell was accomplished by summing the central 12 × 12 population cells.”
    pop_regrid = pop_full.coarsen(lat=12, lon=12).sum()

    pop_regrid.to_netcdf(f"{POP_DIR}ssp2_total_regrid_{year}.nc")

print("Finished processing population regridding")

Processing 2000
Processing 2010
Processing 2020
Processing 2030
Processing 2040
Processing 2050
Processing 2060
Processing 2070
Processing 2080
Processing 2090
Processing 2100
Finished processing population regridding


In [2]:
# Concatenate regridding population data to one file
tot_pop = []
years = range(2000, 2101, 10)

for year in years:
    print(f"Processing {year}")
    pop = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_{year}.nc")
    tot_pop.append(pop)

population_time_series = xr.concat(tot_pop, xr.DataArray(years, dims="year", name="year"))

population_time_series.to_netcdf(f"{POP_DIR}ssp2_total_regrid_{years[0]}-{years[-1]}.nc")

Processing 2000
Processing 2010
Processing 2020
Processing 2030
Processing 2040
Processing 2050
Processing 2060
Processing 2070
Processing 2080
Processing 2090
Processing 2100
